# 分治法：最近点对问题
本次作业将实现课堂上学习的最近点对问题的分治算法。给定一组点，该算法能够找到欧式距离最小的一对点。

## 环境配置说明：
### 如果在本地运行：
#### 首次配置
* 安装Anaconda：访问 https://www.anaconda.com/products/distribution
* 创建conda环境：`conda create -n cs131232 python=3.8`
  * 详见 https://conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html
* 激活环境：`conda activate cs131232`
* 安装pip：`conda install pip`
* 安装Jupyter：`conda install jupyter`
#### 每次启动
* 确保激活环境：`conda activate cs131232`
* 启动Jupyter：`jupyter notebook` 或 `jupyter lab`
* 运行以下单元格**并在需要时重启内核**

In [ ]:
# Install dependencies
!pip install -r requirements.txt --quiet

In [ ]:
import otter

assert (otter.__version__ >= "4.4.1"), "Please reinstall the requirements and restart your kernel."

grader = otter.Notebook("closest-pair.ipynb")
import numpy as np
from time import time
import tqdm
import math
import numpy.random as random

rng_seed = 0

### 问题 1: 距离计算
在实现最近点对算法之前，先创建辅助函数。
> **任务 1:** 编写计算两点间欧氏距离的函数

_分值：_ 5

In [ ]:
def distance(p1, p2):
    """ Finds the distance between point p1 and point p2.

    Args:
        p1 (Tuple[int, int]): A point.
        p2 (Tuple[int, int]): A point.

    Returns:
        double: The distance between point p1 and point p2.
    """
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


In [ ]:
grader.check("q1")

### 问题 2: 暴力解法
现在实现不使用分治法的最近点对算法，要求时间复杂度为$\,O(n^2)\,$。可以使用前一问题的函数。
> **任务 2:** 暴力搜索最近点对

如果有多组解，返回任意一组即可。

_分值：_ 10

In [ ]:
import itertools


def naive_closest_pair(s):
    """Implements the closest pair algorithm to find the pair of points of minimum distance from set s naively.

    Args:
        s ([List[Tuple[int, int]]]): The list of points.

    Returns:
        Tuple[Tuple[int, int], Tuple[int, int]]: A tuple of the closest pair of points.
    """
    points = list(s)
    best_pair, best_dist = None, float("inf")
    # Every pair of points is considered once: O(n^2) distance computations.
    for p1, p2 in itertools.combinations(points, 2):
        d = distance(p1, p2)
        if d < best_dist:
            best_dist, best_pair = d, (p1, p2)
    return best_pair


In [ ]:
grader.check("q2")

### 问题 3: 分治算法
现在实现 $\,O(nlogn)\,$ 时间复杂度的分治算法。可以使用前面问题的函数。
> **任务 3:** 分治法解决最近点对问题

如果有多组解，返回任意一组即可。

本任务分为A、B、C、D四个部分。前三部分（A、B、C）不进行测试，也不计分；最后一部分（D）占30分。如果你愿意，可以直接完成D部分。

#### A 部分：中位点查找
假设点已按 **x** 坐标排好序，<span style="color:blue">find_median</span>算法需找到 **x** 轴的中位点。

_分值：_ 0

In [ ]:
def find_median(s):
    """Finds the median x-coordinate among the points s.

    Args:
        s ([List[Tuple[int, int]]]): The list of points sorted by x-coordinate.

    Returns:
        int: The median x-coordinate.
    """
    # s is already sorted by x-coordinate, so the middle element holds the median.
    return s[len(s) // 2][0]


#### B 部分：带状区域处理
在我们实现递归之前，了解如何计算带状区域中各点之间的距离可能会有所帮助。利用这一部分，我们可以找出带状区域中最近的一对点。你可以假设这些点是按照 **y** 坐标排好序的（这个假设为什么有助于解决问题？）。

_分值：_ 0

In [ ]:
def closest_pair_in_strip(points_in_strip):
    """Finds the pair of points of minimum distance among the points in points_in_strip.

    Args:
        s ([List[Tuple[int, int]]]): The list of points in the strip sorted by y-coordinate.

    Returns:
        Tuple[Tuple[int, int], Tuple[int, int]]: A tuple of the closest pair of points in the strip.
    """
    points = list(points_in_strip)
    best_pair, best_dist2 = None, float("inf")
    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            # Sorted by y, so once the vertical gap alone exceeds the best
            # distance found so far, no later point can beat it.
            dy = points[j][1] - points[i][1]
            if dy * dy > best_dist2:
                break
            dist2 = (points[j][0] - points[i][0]) ** 2 + dy * dy
            if dist2 < best_dist2:
                best_dist2, best_pair = dist2, (points[i], points[j])
    return best_pair


#### C 部分：递归实现
现在，我们希望使用前面的函数来编写递归分治算法的主体。该部分从给定的点集中找出距离最近的一对点。

关于输入点的顺序，你应该做出什么假设？为什么递归调用有两个输入？

_分值：_ 0

In [ ]:
def closest_pair_recurse(sx, sy):
    """Implements the recursive process of the divide and conquer approach.

    Args:
        sx ([List[Tuple[int, int]]]): The list of points.
        sy ([List[Tuple[int, int]]]): The list of points.

    Returns:
        Tuple[Tuple[int, int], Tuple[int, int]]: A tuple of the closest pair of points.
    """
    # Both inputs hold the same points: sx sorted by x-coordinate, sy sorted by
    # y-coordinate. Passing both down avoids re-sorting at every level.
    def recurse(sx, sy):
        n = len(sx)
        if n < 2:
            return None, float("inf")
        if n <= 3:
            best_pair, best_dist2 = None, float("inf")
            for i in range(n):
                for j in range(i + 1, n):
                    dist2 = (sx[i][0] - sx[j][0]) ** 2 + (sx[i][1] - sx[j][1]) ** 2
                    if dist2 < best_dist2:
                        best_dist2, best_pair = dist2, (sx[i], sx[j])
            return best_pair, best_dist2

        mid = n // 2
        mid_x = sx[mid][0]
        left_x, right_x = sx[:mid], sx[mid:]

        # Split sy according to which half each point landed in. A counter is
        # used instead of a set so that duplicate points split correctly.
        counts = {}
        for p in left_x:
            counts[p] = counts.get(p, 0) + 1
        left_y, right_y = [], []
        for p in sy:
            c = counts.get(p, 0)
            if c > 0:
                counts[p] = c - 1
                left_y.append(p)
            else:
                right_y.append(p)

        left_pair, left_dist2 = recurse(left_x, left_y)
        right_pair, right_dist2 = recurse(right_x, right_y)
        if left_dist2 <= right_dist2:
            best_pair, best_dist2 = left_pair, left_dist2
        else:
            best_pair, best_dist2 = right_pair, right_dist2

        # Only points within best_dist of the dividing line can improve on the
        # two halves; they are already in y-order, so the inner loop can stop
        # as soon as the vertical gap exceeds best_dist.
        strip = [p for p in sy if (p[0] - mid_x) ** 2 <= best_dist2]
        for i in range(len(strip)):
            for j in range(i + 1, len(strip)):
                dy = strip[j][1] - strip[i][1]
                if dy * dy > best_dist2:
                    break
                dist2 = (strip[j][0] - strip[i][0]) ** 2 + dy * dy
                if dist2 < best_dist2:
                    best_dist2, best_pair = dist2, (strip[i], strip[j])
        return best_pair, best_dist2

    pair, _ = recurse(list(sx), list(sy))
    return pair


#### D 部分
如果你顺利完成了前面的几部分，现在可以将所有内容整合起来，完成算法的实现。如果不打算使用前面部分的子程序，也可以从这里开始进行实现。
> **任务 3:** 分治法解决最近点对问题

如果有多组解，返回任意一组即可。 算法的时间复杂度需为$\,O(nlogn)\,$。

_分值：_ 30

In [ ]:
def closest_pair(s):
    """Implements the closest pair algorithm to find the pair of points of minimum distance from set s.

    Args:
        s ([List[Tuple[int, int]]]): The list of points.

    Returns:
        Tuple[Tuple[int, int], Tuple[int, int]]: A tuple of the closest pair of points.
    """
    points = [(int(p[0]), int(p[1])) for p in s]
    # Sort once by x and once by y; the recursion keeps both orders at all times.
    sx = sorted(points)
    sy = sorted(points, key=lambda p: (p[1], p[0]))
    return closest_pair_recurse(sx, sy)


In [ ]:
grader.check("q6")

## 提交

确保在运行下面的单元格之前，你已经按顺序运行了笔记本中的所有单元格，以便所有图像/图表都能在输出中显示。下面的单元格将为你生成一个用于提交的压缩文件。

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)